## TRABAJO GRUPAL SISTEMA RECOMENDACIONES

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import random

# Inicializar sesión de Spark
spark = SparkSession.builder.appName("RecomendadorALS").getOrCreate()

# 1. Cargar tu CSV (cambia la ruta por la tuya)
# El CSV debe contener al menos: categoria, marca, tienda, precio
df_productos_raw = spark.read.csv("productos.csv", header=True, inferSchema=True)

# 2. Asegurar un ID numérico secuencial para los productos
windowSpec = Window.orderBy("categoria", "marca")
df_productos = df_productos_raw.withColumn("product_id", F.row_number().over(windowSpec))

print(f"Productos cargados: {df_productos.count()}")
df_productos.show(5)

Productos cargados: 35
+-----------+--------------------+-----------------+--------------+-------------------+------+----------+
|id_producto|              nombre|        categoria|         marca|             tienda|precio|product_id|
+-----------+--------------------+-----------------+--------------+-------------------+------+----------+
|         29|Aire Acondicionad...|    Climatizacion|            LG|Creditos Economicos|469.99|         1|
|         30|Ventilador de Ped...|    Climatizacion|        Taurus|Creditos Economicos|  45.0|         2|
|         27|Freidora de Aire ...|Electrodomesticos|Black & Decker|Creditos Economicos|  85.0|         3|
|         28|Cafetera Hamilton...|Electrodomesticos|Hamilton Beach|Creditos Economicos|  55.0|         4|
|         21|Cocina Indurama M...|Electrodomesticos|      Indurama|Creditos Economicos|459.99|         5|
+-----------+--------------------+-----------------+--------------+-------------------+------+----------+
only showing top 5 rows

## SIMULACION DE CLIENTES

In [2]:
# Definición de las reglas de los perfiles de consumo
# Esto nos servirá para calcular la "afinidad de categoría" y "afinidad de marca" más adelante.
perfiles_config = {
    "tecnologico": {"categorias_afines": ["Tecnología", "Electrónica", "Computación"], "marcas_afines": ["Apple", "Samsung", "Asus", "Sony"], "presupuesto_max": 2000},
    "gamer": {"categorias_afines": ["Videojuegos", "Computación", "Consolas"], "marcas_afines": ["Sony", "Nintendo", "Asus", "Razer"], "presupuesto_max": 1500},
    "hogar": {"categorias_afines": ["Línea Blanca", "Hogar", "Muebles"], "marcas_afines": ["Indurama", "LG", "Whirlpool"], "presupuesto_max": 800},
    "estudiante": {"categorias_afines": ["Libros", "Papelería", "Tecnología"], "marcas_afines": ["HP", "Lenovo", "Dell"], "presupuesto_max": 400},
    "fitness": {"categorias_afines": ["Deportes", "Suplementos", "Ropa Deportiva"], "marcas_afines": ["Nike", "Adidas", "Puma"], "presupuesto_max": 300},
    "premium": {"categorias_afines": ["Moda", "Tecnología", "Joyas"], "marcas_afines": ["Apple", "Sony", "Samsung"], "presupuesto_max": 5000},
    "ahorrador": {"categorias_afines": ["Supermercado", "Hogar"], "marcas_afines": ["Genérica", "Maggi", "La Fabril"], "presupuesto_max": 50}
}

lista_perfiles = list(perfiles_config.keys())

# Generar 30 clientes (user_id del 1 al 30)
datos_clientes = []
for user_id in range(1, 31):
    perfil_asignado = random.choice(lista_perfiles)
    datos_clientes.append((user_id, perfil_asignado))

# Crear el DataFrame de clientes en PySpark
df_clientes = spark.createDataFrame(datos_clientes, ["user_id", "perfil"])
df_clientes.show(5)

+-------+-----------+
|user_id|     perfil|
+-------+-----------+
|      1|    premium|
|      2|tecnologico|
|      3|    premium|
|      4|    fitness|
|      5|      hogar|
+-------+-----------+
only showing top 5 rows


## GENERACIÓN DE MATRIZ DE IDENTIDAD

In [3]:
from pyspark.sql.types import FloatType

# Convertimos la configuración de perfiles en un broadcast para usarlo en la lógica de cálculo
perfiles_bc = spark.sparkContext.broadcast(perfiles_config)

def calcular_rating_logica(perfil, categoria, marca, precio):
    config = perfiles_bc.value.get(perfil)

    # 1. Base fija
    base = 2.5

    # 2. Afinidad de categoría (si coincide, suma +1.0)
    afinidad_cat = 1.0 if categoria in config["categorias_afines"] else 0.0

    # 3. Afinidad de marca (si coincide, suma +0.5)
    afinidad_marca = 0.5 if marca in config["marcas_afines"] else 0.0

    # 4. Penalización de precio
    # Si supera el presupuesto del perfil, penaliza severamente. Si no, penaliza levemente por el gasto.
    if precio > config["presupuesto_max"]:
        penalizacion_precio = 1.5
    else:
        penalizacion_precio = (precio / config["presupuesto_max"]) * 0.5

    # 5. Ruido controlado (entre -0.3 y 0.3)
    ruido = random.uniform(-0.3, 0.3)

    # Aplicar fórmula
    rating_final = base + afinidad_cat + afinidad_marca - penalizacion_precio + ruido

    # Limitar el resultado estrictamente entre 1 y 5
    return float(max(1.0, min(5.0, rating_final)))

# Registrar como UDF de PySpark
rating_udf = F.udf(calcular_rating_logica, FloatType())

# Generar combinaciones de interacciones
# Para asegurar al menos 250 interacciones aleatorias distribuidas entre los 30 clientes:
df_interacciones_base = df_clientes.crossJoin(df_productos)

# Calcular el rating para cada combinación
df_matriz_utilidad = df_interacciones_base.withColumn(
    "rating",
    rating_udf(F.col("perfil"), F.col("categoria"), F.col("marca"), F.col("precio"))
)

# El enunciado pide un mínimo de 250 interacciones. Un crossJoin de 30 clientes x 30-50 productos dará ~1000.
# Si deseas simular que no todos compraron todo (escasez), podemos tomar una muestra de, por ejemplo, 400 registros.
df_interacciones_final = df_matriz_utilidad.sample(withReplacement=False, fraction=0.4, seed=42)

# Verificar que cumplimos con el mínimo del entregable
print(f"Total de interacciones simuladas: {df_interacciones_final.count()}")
df_interacciones_final.select("user_id", "product_id", "perfil", "categoria", "rating").show(10)

Total de interacciones simuladas: 406
+-------+----------+-----------+-----------------+---------+
|user_id|product_id|     perfil|        categoria|   rating|
+-------+----------+-----------+-----------------+---------+
|      1|         4|    premium|Electrodomesticos|2.7798557|
|      1|         8|    premium|Electrodomesticos|2.5573912|
|      1|        13|    premium|            Hogar|2.7167346|
|      1|        17|    premium|       TV y Audio|2.1734266|
|      1|        19|    premium|       TV y Audio|2.7900045|
|      1|        26|    premium|       Tecnologia|2.5177538|
|      1|        27|    premium|       Tecnologia|2.4936836|
|      1|        28|    premium|       Tecnologia|2.1870708|
|      1|        32|    premium|       Tecnologia|2.5932212|
|      2|         2|tecnologico|    Climatizacion|2.3738513|
+-------+----------+-----------+-----------------+---------+
only showing top 10 rows


## ENTRENAMIENTO Y VALIDACION  DE MODELO ALS

In [4]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Split de datos: Entrenamiento (80%) y Prueba (20%)
(train, test) = df_interacciones_final.randomSplit([0.8, 0.2], seed=42)

# Configurar el modelo ALS
# coldStartStrategy="drop" asegura que si hay IDs en test que no se vieron en train, no rompa la métrica arrojando NaN
als = ALS(
    userCol="user_id",
    itemCol="product_id",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True
)

# Aquí puedes experimentar iterando con diferentes valores (Hyperparameter tuning)
# Por ejemplo, variando rank=[5, 10] y regParam=[0.01, 0.1]
model = als.setRank(10).setRegParam(0.1).fit(train)

# Predicciones
predictions = model.transform(test)

# Evaluación del modelo usando RMSE
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print(f"Error cuadrático medio (RMSE) en el set de prueba = {rmse}")

evaluator_mae = RegressionEvaluator(metricName="mae", labelCol="rating", predictionCol="prediction")
mae = evaluator_mae.evaluate(predictions)
print(f"Error absoluto medio (MAE) en el set de prueba = {mae}")

Error cuadrático medio (RMSE) en el set de prueba = 0.39719725704239595
Error absoluto medio (MAE) en el set de prueba = 0.3187577884588669


## RECOMENDACIONES PARA 5 CLIENTES

In [6]:
# Generar las 5 mejores recomendaciones para TODOS los usuarios
userRecs = model.recommendForAllUsers(5)

# Filtrar o tomar una muestra de 5 usuarios específicos para el reporte final
df_recomendaciones_5_clientes = userRecs.limit(5)

# Hacer un desglose legible uniendo con el nombre original de tus productos
df_recs_exploded = df_recomendaciones_5_clientes.withColumn("recommendation", F.explode("recommendations")) \
    .select("user_id", F.col("recommendation.product_id").alias("product_id"), F.col("recommendation.rating").alias("predicted_rating"))

# Unir con la tabla de productos para ver qué estamos recomendando realmente
df_informe_final = df_recs_exploded.join(df_productos, on="product_id", how="inner") \
    .select("user_id", "product_id", "categoria", "marca", "precio", "predicted_rating") \
    .orderBy("user_id", F.desc("predicted_rating"))

df_informe_final.show(5, truncate=False)

+-------+----------+----------+--------+-------+----------------+
|user_id|product_id|categoria |marca   |precio |predicted_rating|
+-------+----------+----------+--------+-------+----------------+
|1      |16        |Hogar     |Xtratech|119.99 |2.9427087       |
|1      |15        |Hogar     |Generico|79.99  |2.7749357       |
|1      |12        |Hogar     |Chaide  |289.99 |2.766916        |
|1      |24        |Tecnologia|Apple   |1099.99|2.7264948       |
|1      |13        |Hogar     |Generico|229.99 |2.6993132       |
+-------+----------+----------+--------+-------+----------------+
only showing top 5 rows
